In [13]:
import os
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal
from langchain_openai import ChatOpenAI

In [14]:
class State(TypedDict):
    query: str
    category: Literal["billing", "technical", "general"]
    response: str

In [15]:
model = ChatOpenAI(
    model=os.getenv("GEMINI_AI_MODEL"),
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    temperature=0
)

In [16]:
def classify_query(state: State):
    prompt = f""""     Classify this customer query into exactly one category:
    billing, technical, or general.

    Rules:
    - Billing: payments, refunds, invoices
    - Technical: bugs, errors, login problems
    - General: all other questions

    Return only the category name.

    Customer query: {state['query']} """

    res = model.invoke(prompt).content.strip().lower()
    if res not in {"billing", "technical", "general"}:
        res = "general"  # Default to general if classification fails
    return {"category": res}

def billing_node(state: State):
    res = model.invoke( f"Help the customer with this billing query: {state['query']}").content
    return {"response": res}

def technical_node(state: State):
    res = model.invoke( f"Help troubleshoot this technical issue: {state['query']}").content
    return {"response": res}

def general_node(state: State):
    res = model.invoke( f"Help troubleshoot this technical issue: {state['query']}").content
    return {"response": res}

def route_query(state: State)-> Literal["billing", "technical", "general"]:
    return state["category"]


In [17]:
graph = StateGraph(State)

graph.add_node("classify_query",classify_query)
graph.add_node("billing_node",billing_node)
graph.add_node("technical_node",technical_node)
graph.add_node("general_node",general_node)


graph.add_edge(START, "classify_query")
graph.add_conditional_edges("classify_query",route_query,{
    "billing": "billing_node",
    "technical": "technical_node",
    "general": "general_node"
})
graph.add_edge("billing_node", END)
graph.add_edge("technical_node", END)
graph.add_edge("general_node", END)

workflow = graph.compile()

In [19]:
input_state = {"query": "I paid twice for my subscription. Can I get a refund?"}
output_state = workflow.invoke(input_state)
print(output_state['response'])

Absolutely, I can help you with this! I understand it can be concerning to see a duplicate charge.

To quickly locate both payments and process your refund, could you please provide the following details?

*   **Email address** associated with your subscription:
*   **Name of the subscription** (e.g., Premium Plan, Pro Membership):
*   **Dates and approximate times** of both payments:
*   **Payment methods used** (e.g., Visa ending in 1234, PayPal, Mastercard):
*   **Transaction IDs or reference numbers** for both payments (if you have them from your bank statement or payment confirmation emails):

Once I have this information, I will verify the duplicate payment in our system and initiate a refund for one of the charges. Refunds typically take **5-10 business days** to appear back on your original payment method, depending on your bank or card issuer.

Please provide these details, and I'll get this sorted for you right away.
